# Challenge 7a: Variable Star Classification from Light Curve Spectra

**Author:** Dr Rob Lyon

**Version:** 1.0

## Code & License
Copyright &copy; 2026 Robert Lyon. All Rights Reserved.

Permission is granted to use, copy, and modify this material for
personal educational purposes only. Redistribution, resale, or use
in commercial or institutional settings requires prior written
permission from the author. This material is provided for educational
purposes as part of a structured course. The author accepts no
liability for incorrect results or decisions arising from use of this
material. All datasets used in this challenge are synthetic and do
not represent proprietary or confidential experimental data.

## Table of Contents

1. [Introduction](#1-introduction)
2. [Useful Links](#2-useful-links)
3. [Libraries and Environment Setup](#3-libraries-and-environment-setup)
   - [3.1 Working Environment](#31-working-environment)
   - [3.2 Libraries Used](#32-libraries-used)
   - [3.3 Data](#33-data)
   - [3.4 Imports](#34-imports)
4. [The Data](#4-the-data)
   - [4.1 Loading the Data](#41-loading-the-data)
   - [4.2 Understanding the Features](#42-understanding-the-features)
   - [4.3 Exploring the Data](#43-exploring-the-data)
   - [4.4 Preprocessing](#44-preprocessing)
5. [The Challenge](#5-the-challenge)
   - [5.1 Why Scaling Matters for SVMs](#51-why-scaling-matters-for-svms)
   - [5.2 Training the Model](#52-training-the-model)
   - [5.3 Evaluating the Model](#53-evaluating-the-model)
   - [5.4 Interpretation](#54-interpretation)
   - [5.5 Reflection Questions](#55-reflection-questions)
6. [Solution](#6-solution)
7. [References](#7-references)

---
## 1. Introduction

Variable stars change in brightness over time. This variability can arise from several physical mechanisms; such as pulsation, rotation, magnetic activity, or binary interaction, but for the classes in this challenge the dominant mechanism is **pulsation**. In these stars, the outer layers periodically expand and contract due to internal pressure and gravity not being in perfect equilibrium. These oscillations change both the star’s radius and surface temperature, which in turn modulates its luminosity (intrinsic brightness).

Different pulsation modes, occurring in stars with different masses, ages, and evolutionary stages, produce recognisably different **light curve** shapes and periods. A light curve is simply a time series showing how a star’s brightness changes over time. These patterns encode rich physical information about the star’s interior structure.

Classifying variable stars is scientifically important. For example:

- **Cepheid variables** exhibit a well-defined relationship between their pulsation period and intrinsic luminosity, making them **standard candles** used to measure astronomical distances and calibrate the cosmic distance ladder.
- **RR Lyrae stars** are older, low-mass pulsating stars that trace ancient stellar populations and are widely used to map the structure of the Milky Way halo.
- **Delta Scuti** and **Beta Cephei stars** oscillate in ways that allow astrophysicists to probe internal stellar structure through a technique called **asteroseismology**, essentially “listening” to stellar vibrations to infer internal composition and dynamics.

Modern sky surveys make manual classification impossible at scale. The **Optical Gravitational Lensing Experiment (OGLE)** has catalogued over 450,000 variable stars in the Magellanic Clouds and Galactic bulge, while the **Gaia Data Release 3 (Gaia DR3)** provides variability classifications for over 10.5 million sources across the sky. These large-scale surveys rely heavily on automated machine learning classifiers trained on features extracted from light curves and low-resolution photometric spectra. Correct classification—especially for rare classes and objects near class boundaries—remains an active research problem with direct consequences for precision cosmology and stellar astrophysics.

The **Spectra** dataset contains 10,000 stellar observations spanning seven variable star classes. The features are derived from light curve statistics and survey-quality photometric measurements, as well as low-resolution spectral information. Importantly, two of the features are expressed in physical units that differ by several orders of magnitude from the remaining dimensionless or normalised indices. This is not an inconsistency, but a reflection of real astronomical survey data: pulsation periods may be recorded in minutes or days, while luminosities may be expressed in solar units or astronomical magnitudes.

This distinction in scale is crucial. As you will see, differences in feature magnitude can significantly affect distance-based methods such as Support Vector Machines (SVMs), particularly when features are not properly normalised.

This challenge is deliberately designed so that a correctly implemented SVM achieves only moderate accuracy. This is not a failure of implementation, but rather a reflection of two intrinsic properties of the problem: the astrophysical degeneracy between certain class pairs (different physical processes producing similar observable signals), and the structural limitations of kernel SVMs when faced with datasets containing small disjuncts and heavily imbalanced minority classes embedded within a dominant majority class. The reflection questions at the end of the notebook encourage you to interpret these limitations carefully and consider what they imply about model choice and feature representation.

### Learning Objectives

- Apply `SVC` with RBF and linear kernels to a seven-class astrophysical
  classification problem with severe feature scale heterogeneity
- Demonstrate and quantify the effect of `StandardScaler` on SVM
  performance by comparing scaled and unscaled results explicitly
- Evaluate a multiclass classifier on an imbalanced seven-class dataset
  using per-class precision, recall, and macro-averaged F1 score
- Conduct a kernel and C hyperparameter sweep and select a model based
  on test performance
- Identify small disjuncts and rare classes in the confusion matrix and
  reason about why SVMs struggle with them
- Reflect honestly on the limits of the chosen algorithm and articulate
  what alternative approaches might do better

---
## 2. Useful Links

| Resource | URL |
|---|---|
| `SVC` (scikit-learn) | https://scikit-learn.org/stable/modules/generated/sklearn.svm.SVC.html |
| SVM user guide (scikit-learn) | https://scikit-learn.org/stable/modules/svm.html |
| `StandardScaler` (scikit-learn) | https://scikit-learn.org/stable/modules/generated/sklearn.preprocessing.StandardScaler.html |
| `classification_report` (scikit-learn) | https://scikit-learn.org/stable/modules/generated/sklearn.metrics.classification_report.html |
| `ConfusionMatrixDisplay` (scikit-learn) | https://scikit-learn.org/stable/modules/generated/sklearn.metrics.ConfusionMatrixDisplay.html |
| OGLE variable star catalogues | https://ogle.astrouw.edu.pl |
| Gaia DR3 variability classification (Eyer et al. 2023) | https://doi.org/10.1051/0004-6361/202244242 |
| Delta Scuti / SX Phe distinction (McNamara 2011) | https://doi.org/10.1088/0004-637X/736/2/69 |
| W Virginis vs Cepheid classification (Beaulieu et al. 2001) | https://doi.org/10.1051/0004-6361:20010993 |

---
## 3. Libraries and Environment Setup

### 3.1 Working Environment

This notebook requires Python 3.9 or later and scikit-learn 1.0 or later.
SVM training on 10,000 samples with RBF kernel can take 30-120 seconds
per fit depending on hardware and the value of C. The hyperparameter sweep
in Section 5 runs multiple fits; budget 5-15 minutes for the full sweep.

### 3.2 Libraries Used

| Library | Purpose |
|---|---|
| `numpy` | Numerical operations |
| `pandas` | Loading and inspecting the dataset |
| `matplotlib` | Plotting |
| `seaborn` | Distribution plots and heatmaps |
| `sklearn.model_selection` | Stratified train/test splitting |
| `sklearn.preprocessing` | `StandardScaler` for feature normalisation |
| `sklearn.svm` | `SVC` classifier |
| `sklearn.metrics` | Confusion matrix, classification report, accuracy |

### 3.3 Data

| Property | Value |
|---|---|
| Filename | `spectra.csv` |
| Location | `data/spectra.csv` (relative to this notebook) |
| Total rows | 10,000 |
| Features | 9 columns (mixed physical units and dimensionless indices) |
| Target column | `star_type` |
| Class 0 | delta\_scuti: ~2,730 samples (~27.3%) |
| Class 1 | rr\_lyrae: ~2,164 samples (~21.6%) |
| Class 2 | cepheid: ~1,817 samples (~18.2%) |
| Class 3 | mira: ~1,144 samples (~11.4%) |
| Class 4 | w\_virginis: ~975 samples (~9.8%) |
| Class 5 | beta\_cephei: ~726 samples (~7.3%) |
| Class 6 | sx\_phoenicis: ~444 samples (~4.4%) |

The class distribution spans a 6:1 ratio from majority (delta\_scuti) to
minority (sx\_phoenicis). Two features (`period_min` and `luminosity_lsun`)
have ranges that span several orders of magnitude, while the remaining
seven features are dimensionless or in narrow physical ranges.

### 3.4 Imports

In [ ]:
# Listing 1.
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
    ConfusionMatrixDisplay,
)

CLASS_NAMES = ['delta_scuti', 'rr_lyrae', 'cepheid', 'mira',
               'w_virginis', 'beta_cephei', 'sx_phoenicis']
FEATURE_COLS = ['period_min', 'amplitude_mag', 'bp_rp', 'phi21', 'r21',
                'fe_h_proxy', 'skewness', 'luminosity_lsun', 'n_modes']

print('Libraries loaded successfully.')

---
## 4. The Data

### 4.1 Loading the Data

In [ ]:
# Listing 2.
df = pd.read_csv('data/spectra.csv')

print(f'Dataset shape: {df.shape}')
print(f'Columns: {df.columns.tolist()}')
print()
df.head(10)

### 4.2 Understanding the Features

Nine features represent quantities measurable from photometric surveys and
low-resolution spectra. Two are in physical units with very wide dynamic
ranges; the remaining seven are dimensionless or in narrow ranges.

| Feature | Units | Approx range | Description |
|---|---|---|---|
| `period_min` | minutes | 10 to 600,000 | Dominant pulsation period. Short-period classes (Delta Scuti, Beta Cephei, SX Phe): 10-900 min. RR Lyrae: 700-1800 min. Cepheid/W Vir: 300-115,000 min. Mira: 130,000-600,000 min. Spans ~4 orders of magnitude. |
| `amplitude_mag` | magnitudes | 0.001 to 13 | Peak-to-peak V-band amplitude. Miras: large (1-12 mag). RR Lyrae: 0.3-1.8 mag. Cepheids: 0.1-2.5 mag. Delta Scuti: very small (0.001-1 mag). |
| `bp_rp` | mag | -0.1 to 5.7 | Gaia BP-RP colour index. Proxy for effective temperature. Hot B stars (Beta Cephei): 0.1-0.8. A/F stars (Delta Scuti, SX Phe): 0.6-1.3. G/K giants (Cepheid, W Vir, RR Lyrae): 0.9-2.0. Cool AGB giants (Mira): 2.0-5.0. |
| `phi21` | radians | 0 to 8.5 | Fourier phase parameter phi\_21 = phi\_2 - 2*phi\_1. Encodes light curve shape beyond amplitude and period. Derived from Fourier decomposition of the phased light curve. |
| `r21` | dimensionless | 0.01 to 0.95 | Fourier amplitude ratio R\_21 = A\_2/A\_1. Second-to-first harmonic amplitude ratio. Related to light curve asymmetry. |
| `fe_h_proxy` | dex | -3.2 to 0.9 | Iron abundance [Fe/H] proxy from spectral line ratios or photometric colour calibrations. Population I stars (Cepheid, Delta Scuti, Beta Cephei): near-solar (~0). Population II stars (RR Lyrae, W Vir, SX Phe): metal-poor (-0.4 to -2.5). |
| `skewness` | dimensionless | -1.5 to 2.0 | Skewness of the phased light curve flux distribution. RR Lyrae: strongly asymmetric (fast rise, slow fall). Mira: also asymmetric. Delta Scuti and Beta Cephei: near-sinusoidal, near zero. |
| `luminosity_lsun` | L\_sun | 0.5 to 500,000 | Luminosity in solar units, derived from absolute magnitude. Main-sequence A/F stars: 3-80 L\_sun. Horizontal branch RR Lyrae: 40-80 L\_sun. B supergiants (Beta Cephei): 1,000-100,000 L\_sun. Classical supergiants (Cepheid, W Vir): 1,000-80,000 L\_sun. AGB (Mira): 5,000-200,000 L\_sun. Spans ~5 orders of magnitude. |
| `n_modes` | log10(count) | -0.3 to 2.2 | Number of independently detected pulsation frequencies, in log10 scale. Delta Scuti and SX Phe: multi-mode, typically 0.7-1.7. RR Lyrae and Mira: few modes, 0-0.5. Cepheid and W Vir: 0-0.6. Beta Cephei: intermediate, 0.2-1.0. |

### 4.3 Exploring the Data

In [ ]:
# Listing 3.
print('Class distribution:')
counts = df['star_type'].value_counts().sort_index()
for label, count in counts.items():
    print(f'  Class {label} ({CLASS_NAMES[label]:14s}): {count:5d}  ({100*count/len(df):.1f}%)')

print()
print('Feature value ranges:')
print(df[FEATURE_COLS].describe().loc[['min','max','std']].round(2).to_string())

**Questions to consider:**

- The ranges of `period_min` and `luminosity_lsun` span several orders of
  magnitude. How large is the ratio of their standard deviations relative
  to the standard deviations of the dimensionless features like `phi21` or
  `r21`? What does this imply for an RBF kernel SVM that uses Euclidean
  distance in feature space?
- The SX Phoenicis class has only 444 samples compared to 2,730 Delta Scuti.
  This 6:1 ratio matters for an SVM: support vectors are found near the
  decision boundary, and the SVM will have fewer SX Phe support vectors to
  define that boundary. What do you expect this to mean for SX Phe recall?
- There are 975 W Virginis and 1,817 Cepheid stars. Both follow
  period-luminosity relations. Before looking at the feature distributions,
  which features do you expect to be most useful for separating them?

In [ ]:
# Listing 4.
# Distributions for the two physical-unit features
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
colours = {0:'steelblue', 1:'darkorange', 2:'seagreen', 3:'firebrick',
           4:'purple', 5:'saddlebrown', 6:'crimson'}

for ax, feat in zip(axes, ['period_min', 'luminosity_lsun']):
    for cls in range(7):
        subset = df.loc[df['star_type'] == cls, feat].dropna()
        # Log scale for these features
        np.log10(subset.clip(lower=0.1)).plot(
            kind='kde', ax=ax, label=CLASS_NAMES[cls],
            color=colours[cls], linewidth=1.8
        )
    ax.set_title(f'log10({feat}) by class', fontsize=11)
    ax.set_xlabel(f'log10({feat})')
    ax.legend(fontsize=8)

plt.suptitle('Wide-Range Features: Log Scale Distributions', fontsize=13, y=1.01)
plt.tight_layout()
plt.show()

**Questions to consider:**

- On a log scale, `period_min` shows reasonable class separation. But an
  SVM uses raw Euclidean distance, not log-distance. A Mira star with
  period 400,000 minutes and a Cepheid with period 1,000 minutes differ
  by 399,000 in raw units. A Delta Scuti and Beta Cephei with periods
  of 50 and 200 minutes differ by only 150. The RBF kernel's gamma
  parameter is chosen relative to the total feature variance. What
  happens to the Delta Scuti / Beta Cephei distinction in this geometry?
- `luminosity_lsun` similarly has a huge range. Miras at 100,000 L\_sun
  and Delta Scuti at 10 L\_sun dominate the feature space. What happens
  to the RR Lyrae / Cepheid distinction (which differs by ~50-2,000 L\_sun)
  when Miras are at 100,000 L\_sun in the same space?

In [ ]:
# Listing 5.
# Distributions for the index features
fig, axes = plt.subplots(2, 4, figsize=(18, 8))
axes = axes.flatten()
index_feats = ['amplitude_mag', 'bp_rp', 'phi21', 'r21',
               'fe_h_proxy', 'skewness', 'n_modes']

for ax, feat in zip(axes, index_feats):
    for cls in range(7):
        subset = df.loc[df['star_type'] == cls, feat].dropna()
        subset.plot(kind='kde', ax=ax, label=CLASS_NAMES[cls],
                    color=colours[cls], linewidth=1.8)
    ax.set_title(feat, fontsize=10)
    ax.set_xlabel('')
    ax.legend(fontsize=7)

axes[-1].set_visible(False)
plt.suptitle('Index Feature Distributions by Star Type', fontsize=13, y=1.01)
plt.tight_layout()
plt.show()

**Questions to consider:**

- `fe_h_proxy` shows clear separation between Population I (Delta Scuti,
  Cepheid, Beta Cephei at near-solar) and Population II (RR Lyrae,
  W Virginis, SX Phe at metal-poor). But the distributions overlap: some
  metal-poor Cepheids reach [Fe/H] ~ -1, and some metal-rich RR Lyrae
  reach [Fe/H] ~ -0.3. What does this tell you about the informativeness
  but not sufficiency of metallicity as a discriminator?
- The Delta Scuti and SX Phoenicis distributions look very similar in
  `bp_rp`, `phi21`, `r21`, `skewness`, and `n_modes`. What is the only
  feature that clearly separates them? How wide is the overlap in that
  feature?
- The Cepheid and W Virginis distributions overlap in almost every feature
  shown here. Which features show any separation at all between these two
  classes?

In [ ]:
# Listing 6.
# Correlation matrix
corr = df[FEATURE_COLS].corr()

fig, ax = plt.subplots(figsize=(10, 8))
sns.heatmap(corr, annot=True, fmt='.2f', cmap='coolwarm',
            vmin=-1, vmax=1, linewidths=0.5, ax=ax, square=True)
ax.set_title('Feature Correlation Matrix (Spectra)', fontsize=12)
plt.tight_layout()
plt.show()

**Questions to consider:**

- `period_min` and `luminosity_lsun` are correlated with each other and
  with `amplitude_mag`. This reflects the real period-luminosity relation:
  more luminous pulsators tend to have longer periods and larger amplitudes.
  What consequence does this correlation have for the information content
  of the feature set?
- `fe_h_proxy` and `n_modes` are relatively uncorrelated with the period
  and luminosity features. This means they add independent information.
  `fe_h_proxy` is the metallicity discriminator for the Pop I / Pop II
  confusion pairs. `n_modes` distinguishes multi-mode Delta Scuti and SX
  Phe from single-mode classes like RR Lyrae and Cepheid.
- `bp_rp` is correlated with `period_min` and `luminosity_lsun`. Can you
  explain this physically? (Hint: longer-period pulsators are generally
  more evolved, cooler, and more luminous.)

### 4.4 Preprocessing

As covered in Notebook_7, SVMs find the maximum-margin hyperplane in
feature space. The margin is defined in terms of Euclidean distance.
When one feature has a standard deviation of 116,000 (period\_min) and
another has a standard deviation of 0.21 (r21), the Euclidean distance
between any two points is dominated almost entirely by the period
feature. The RBF kernel computes exp(-gamma * ||x - x'||^2): with a
large ||x - x'||^2 dominated by period, the kernel value is near zero
for almost all pairs, and the SVM degenerates to a period-only
classifier. Scaling removes this pathology.

`StandardScaler` transforms each feature to zero mean and unit variance.
After scaling, all nine features contribute comparably to the distance
computation and the kernel can exploit the full feature set.

The scaling rule: fit `StandardScaler` on training data only. Apply the
training mean and standard deviation to transform both the training and
test sets. This prevents the test set statistics from influencing the
scaling parameters.

In [ ]:
# Listing 7.
# TODO: Implement preprocessing.
#
# 1. Separate features (X) from the target column ('star_type') (y).
#    Use all nine columns in FEATURE_COLS.
#
# 2. Split into training and test sets.
#    Use an 80/20 split, random_state=42, stratify=y.
#    Print class counts in y_train and y_test to confirm stratification
#    has preserved the class ratios. Pay attention to SX Phoenicis (class 6):
#    how many training examples does the minority class have?
#
# 3. Apply StandardScaler.
#    Fit on X_train only, transform both X_train and X_test.
#    Print the mean and standard deviation of period_min and luminosity_lsun
#    learned from the training set (scaler.mean_[0] and scaler.scale_[0]).
#    These will be in the tens of thousands, which illustrates how extreme
#    the raw scales are.

# YOUR CODE HERE

---
## 5. The Challenge

### 5.1 Why Scaling Matters for SVMs

Before training a full model, demonstrate the scaling effect explicitly.
Train an SVM on the **unscaled** data first. Record the test accuracy.
Then train the same SVM on the **scaled** data. The difference will be
large and it will motivate every subsequent modelling decision.

In [ ]:
# Listing 8.
# TODO: Demonstrate the scaling effect.
#
# 1. Train an SVC with kernel='rbf', C=1.0, random_state=42 on the
#    UNSCALED training data (X_train, not X_train_scaled).
#    Predict on the UNSCALED test set (X_test).
#    Print the overall accuracy.
#    Print the classification report with target_names=CLASS_NAMES.
#    Note which classes the model gets right and which it gets wrong.
#    Can you explain the pattern? (Think about which classes are separated
#    by period alone.)
#
# 2. Train the same SVC on the SCALED training data.
#    Predict on the SCALED test set.
#    Print the overall accuracy and classification report.
#
# 3. Print the accuracy gap: scaled_acc - unscaled_acc.
#    This number should be substantial. If it is not, recheck your
#    preprocessing: the scaler must have been fit on the training data
#    before this cell runs.

# YOUR CODE HERE

### 5.2 Training the Model

With the scaling effect established, explore the kernel and C hyperparameter
space on the scaled data. SVMs have two primary hyperparameters:

- **Kernel**: `'linear'` finds a linear decision boundary in feature space.
  `'rbf'` maps data into a higher-dimensional space and finds a linear
  boundary there, allowing non-linear class boundaries in the original
  space. For data with complex, overlapping class regions, `'rbf'` usually
  outperforms `'linear'`.
- **C**: the regularisation parameter. A large C penalises misclassifications
  heavily and produces a tighter margin (lower bias, higher variance). A
  small C allows more misclassifications for a wider margin (higher bias,
  lower variance). The right value depends on the noise level and class
  overlap in the data.

Note that SVM training time scales roughly as O(n^2) to O(n^3) in the
number of training samples. With 8,000 training samples and an RBF kernel,
each fit may take 30-90 seconds. Plan accordingly.

In [ ]:
# Listing 9.
# TODO: Hyperparameter sweep.
#
# Loop over kernels ['linear', 'rbf'] and C values [0.1, 1.0, 10.0].
# For each combination:
#   - Train SVC with random_state=42 on scaled training data.
#   - Compute test accuracy on scaled test data.
#   - Store the result.
# Print a table of results: kernel, C, accuracy.
# Identify which combination gives the best test accuracy.
#
# Note: set max_iter=5000 in SVC to avoid very long fits for high C values.
# If you see a ConvergenceWarning, it means the solver needed more iterations;
# you can try max_iter=10000 but expect longer training time.

# YOUR CODE HERE

In [ ]:
# Listing 10.
# TODO: Train the final model.
#
# Using the best kernel and C from the sweep above, train a final SVC
# on the scaled training data.
# Save it as clf_final for use in Sections 5.3 and 5.4.

# YOUR CODE HERE

### 5.3 Evaluating the Model

With seven classes and a 27:22:18:11:10:7:4 distribution, macro-averaged
F1 score (unweighted mean of per-class F1) is the most informative
headline metric. Overall accuracy is dominated by the performance on
Delta Scuti and RR Lyrae (the two largest classes) and hides poor
performance on W Virginis, Beta Cephei, and SX Phoenicis.

The confusion matrix for seven classes is large. Read it systematically:
for each true class (row), identify which predicted class (column) receives
the most off-diagonal mass. This tells you not just that the model is
wrong, but which class it is confusing each type with.

In [ ]:
# Listing 11.
# TODO: Evaluate the final model.
#
# 1. Generate predictions on the scaled test set.
#
# 2. Print overall accuracy and macro-averaged F1 score.
#    (Use classification_report with output_dict=True to extract macro F1.)
#
# 3. Print the full classification report with target_names=CLASS_NAMES.
#    Focus on the three minority/overlap classes:
#      - w_virginis: what is its recall? Where are the misclassifications going?
#      - beta_cephei: same.
#      - sx_phoenicis: same.
#
# 4. Plot the confusion matrix.
#    Use ConfusionMatrixDisplay.from_predictions() with display_labels=CLASS_NAMES.
#    Rotate the x-axis tick labels for readability: add
#    plt.xticks(rotation=45, ha='right') after the display call.
#    Identify the three largest off-diagonal entries and explain each one
#    in terms of the feature distributions you saw in Section 4.3.

# YOUR CODE HERE

### 5.4 Interpretation

An SVM does not produce feature importances in the same way a decision
tree does. However, you can gain insight by examining the number of
support vectors per class and by testing how performance changes when
features are removed. Here we examine the support vectors.

The number of support vectors for a class reflects how hard the boundary
is to define: a class with many support vectors sits close to other
classes in feature space and requires many points to characterise the
margin. A class with few support vectors has a clean, wide margin from
its neighbours.

In [ ]:
# Listing 12.
# TODO: Examine support vectors.
#
# 1. Print the number of support vectors per class using clf_final.n_support_.
#    The order corresponds to clf_final.classes_.
#    Create a bar chart: x-axis = class names, y-axis = number of support vectors.
#    Title: 'Support vectors per class'.
#
# 2. Which class has the most support vectors?
#    Which has the fewest?
#    Is the pattern consistent with what you observed in the confusion matrix?
#    (Classes with poor recall should have more support vectors because their
#    boundary is ambiguous and requires many examples to define it.)
#
# 3. Print the total number of support vectors as a fraction of the training
#    set size. A high fraction indicates the problem is hard and the margin
#    is narrow.

# YOUR CODE HERE

### 5.5 Reflection Questions

Work through these carefully. They ask you to reason about what the
results mean, not just report them.

1. The SVM achieves around 80% accuracy overall, but macro-averaged F1 is
   substantially lower. Why is there a gap between accuracy and macro F1
   in this specific dataset? Which classes drive this gap?

2. The W Virginis class has a recall of roughly 45-55%, meaning the model
   misclassifies around half of all W Virginis stars. Looking at the
   confusion matrix and the feature distributions, where are those
   misclassifications going? Explain this in terms of the astrophysical
   relationship between W Virginis (Type II Cepheids) and classical Cepheids.

3. SX Phoenicis stars are metal-poor Delta Scuti analogues. The KDE plots
   in Section 4.3 show that `fe_h_proxy` is the primary discriminator,
   but it has substantial measurement scatter. If the scatter were halved
   (better spectroscopic observations), would you expect SX Phe recall
   to improve significantly? What else limits it?

4. The unscaled SVM achieved around 44-56% accuracy. The majority class
   (Delta Scuti) has 27% of samples. A classifier that predicted Delta
   Scuti for everything would achieve 27% accuracy. But the unscaled SVM
   scored higher than 27%. What is the unscaled SVM actually learning,
   if not from the dimensionless index features?

5. The performance ceiling for this problem appears to be around 80-83%
   for a well-tuned SVM. Two paths forward are discussed in the astronomy
   literature for exactly this type of problem. First, imbalance treatment:
   class weights, oversampling of minority classes, or ensemble methods
   designed for imbalanced data (see Notebook_7\_Bonus.ipynb for these
   approaches applied to SVMs). Second, alternative algorithms: random
   forests, gradient boosted trees, and neural networks all handle small
   disjuncts and class overlap differently from SVMs and often outperform
   SVMs on datasets with this structure. Without implementing these, which
   specific failure modes of the SVM do you think a random forest would
   handle better, and why?

---
## 6. Solution

**Read this section only after attempting the challenge yourself.**
Every decision is explained with the reasoning behind it. The cells
run from top to bottom without depending on any variables from Section 5.

### Step 1: Preprocessing

The 80/20 split with stratification is essential here: without stratification,
the 444 SX Phoenicis samples could end up with as few as 70-80 in the test set
by random chance, making the per-class metrics for that class unreliable.
Stratification guarantees the 4.4% fraction is preserved in both splits.

In [ ]:
# Listing 13.
X = df[FEATURE_COLS].values
y = df['star_type'].values

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

scaler = StandardScaler()
X_train_sc = scaler.fit_transform(X_train)
X_test_sc  = scaler.transform(X_test)

print(f'Training set: {X_train_sc.shape}  Test set: {X_test_sc.shape}')
print()
print('Class counts in training set:')
for cls in range(7):
    n = int(np.sum(y_train == cls))
    print(f'  {CLASS_NAMES[cls]:14s} (class {cls}): {n:5d}  ({100*n/len(y_train):.1f}%)')

print(f'\nperiod_min: training mean={scaler.mean_[0]:.1f}, std={scaler.scale_[0]:.1f}')
print(f'luminosity_lsun: training mean={scaler.mean_[7]:.1f}, std={scaler.scale_[7]:.1f}')

The training means and standard deviations for `period_min` and
`luminosity_lsun` are in the tens of thousands. Dividing by these
values brings both features to the same unit-variance scale as `r21`
and `phi21`. Without this step, those two features would numerically
dominate every distance calculation in the SVM.

### Step 2: The Scaling Effect

We train the same RBF SVM on unscaled and scaled data to demonstrate
the consequence of the unit mismatch. This is the most important result
in the challenge: not the final model accuracy, but the gap between
unscaled and scaled performance.

In [ ]:
# Listing 14.
# Unscaled SVM
clf_raw = SVC(kernel='rbf', C=1.0, random_state=42, max_iter=5000)
clf_raw.fit(X_train, y_train)
yp_raw = clf_raw.predict(X_test)
acc_raw = accuracy_score(y_test, yp_raw)

print(f'Unscaled SVM accuracy: {acc_raw:.4f}')
print()
print('Unscaled classification report:')
print(classification_report(y_test, yp_raw, target_names=CLASS_NAMES))

The unscaled SVM typically achieves 44-56% accuracy. The report reveals
that it effectively classifies only Mira (which is separated by period
alone: no other class has periods above 100,000 minutes) and perhaps
RR Lyrae and Delta Scuti (which occupy different ends of the period
range). The intermediate and overlapping classes, Cepheid, W Virginis,
Beta Cephei, and SX Phoenicis, are largely collapsed into whichever
nearby majority class dominates the period neighbourhood.

This is what it means for `period_min` to dominate the kernel: the SVM
can only resolve class differences that manifest as large period
differences. All the information in `fe_h_proxy`, `phi21`, `bp_rp`, and
`n_modes` is essentially invisible because those features contribute
negligible variance to the distance metric.

In [ ]:
# Listing 15.
# Scaled SVM
clf_sc = SVC(kernel='rbf', C=1.0, random_state=42, max_iter=5000)
clf_sc.fit(X_train_sc, y_train)
yp_sc = clf_sc.predict(X_test_sc)
acc_sc = accuracy_score(y_test, yp_sc)

print(f'Scaled SVM accuracy:   {acc_sc:.4f}')
print(f'Accuracy gain from scaling: {acc_sc - acc_raw:.4f}')
print()
print('Scaled classification report:')
print(classification_report(y_test, yp_sc, target_names=CLASS_NAMES))

The accuracy gain from scaling is typically 25-35 percentage points.
This is the direct consequence of allowing all nine features to contribute
to the distance calculation. The improvement is unevenly distributed:
Mira stays high (it was already separated by period), but Beta Cephei
and SX Phoenicis improve dramatically because the now-normalised
`bp_rp`, `fe_h_proxy`, and `n_modes` features can contribute.

### Step 3: Hyperparameter Sweep

In [ ]:
# Listing 16.
results = {}
for kernel in ['linear', 'rbf']:
    for C in [0.1, 1.0, 10.0]:
        clf = SVC(kernel=kernel, C=C, random_state=42, max_iter=5000)
        clf.fit(X_train_sc, y_train)
        yp = clf.predict(X_test_sc)
        acc = accuracy_score(y_test, yp)
        results[(kernel, C)] = acc
        print(f'  kernel={kernel:6s}  C={C:5.1f}  acc={acc:.4f}')

best_key = max(results, key=results.get)
print(f'\nBest configuration: kernel={best_key[0]}, C={best_key[1]}  '
      f'(acc={results[best_key]:.4f})')

The sweep shows that accuracy is relatively stable across kernels and C
values, varying by around 2-4 percentage points. This is informative:
it tells you the performance ceiling for this algorithm on this dataset
is around 80-83%, regardless of how you tune it. The problem is not
that we have the wrong hyperparameters; the problem is that several class
pairs are genuinely hard to separate with a global kernel classifier.

The linear kernel often performs comparably to RBF here. This is
consistent with the feature distributions: most of the class boundaries
in the scaled feature space are approximately linear (the cleaner classes
like Mira, RR Lyrae, and Delta Scuti are separated by thresholds on
individual features). The classes that need non-linear boundaries
(Cepheid/W Vir, Delta Scuti/SX Phe) are exactly the ones the SVM
struggles with regardless of kernel.

### Step 4: Final Model Evaluation

In [ ]:
# Listing 17.
best_kernel, best_C = best_key
clf_final = SVC(kernel=best_kernel, C=best_C, random_state=42, max_iter=5000)
clf_final.fit(X_train_sc, y_train)
y_pred = clf_final.predict(X_test_sc)

report = classification_report(y_test, y_pred, target_names=CLASS_NAMES,
                                output_dict=True)
macro_f1 = report['macro avg']['f1-score']
acc_final = accuracy_score(y_test, y_pred)

print(f'Overall accuracy:    {acc_final:.4f}')
print(f'Macro-averaged F1:   {macro_f1:.4f}')
print()
print(classification_report(y_test, y_pred, target_names=CLASS_NAMES))

The gap between accuracy (~81%) and macro F1 (~75%) reflects the seven-class
imbalance: accuracy is pulled toward the larger classes (Delta Scuti,
RR Lyrae), while macro F1 gives equal weight to every class including the
poorly-classified minorities.

Reading the per-class results:
- **Mira**: near-perfect. The long period and extreme colour (bp\_rp > 2)
  make it the easiest class in the dataset. No other class occupies its
  region of feature space.
- **Delta Scuti, RR Lyrae**: good (88-93% recall). Both classes have
  enough samples and enough separation in period and metallicity that the
  SVM finds a reasonable boundary.
- **Cepheid**: moderate (80-85% recall). Some Cepheid observations are
  misclassified as W Virginis because the period-amplitude distributions
  are deliberately shared.
- **W Virginis**: poor (45-55% recall). The majority of W Vir observations
  are classified as Cepheid. This is the most consequential failure:
  W Vir stars are misidentified as Cepheids, and Cepheids are the primary
  distance indicators in cosmology. A survey-quality classifier that
  confuses 45% of W Vir as Cepheid would systematically bias distances.
- **Beta Cephei**: moderate-poor (50-60% recall). Classified as Delta
  Scuti because both occupy the same period range and the SVM cannot
  reliably use bp\_rp (too noisy) or luminosity (too scattered) to
  distinguish them.
- **SX Phoenicis**: poor (45-55% recall). Almost entirely absorbed into
  Delta Scuti. With 4% of the training set and nearly identical pulsation
  features, the SVM assigns very low probability to SX Phe predictions.

In [ ]:
# Listing 18.
fig, ax = plt.subplots(figsize=(9, 7))
ConfusionMatrixDisplay.from_predictions(
    y_test, y_pred,
    display_labels=CLASS_NAMES,
    cmap='Blues',
    ax=ax
)
plt.xticks(rotation=45, ha='right')
ax.set_title(f'Confusion Matrix: Variable Star Classification\n'
             f'(SVC {best_kernel} kernel, C={best_C})', fontsize=11)
plt.tight_layout()
plt.show()

The confusion matrix makes the class structure visible at a glance.
The diagonal is dominated by the large, clean classes (Delta Scuti, RR Lyrae,
Cepheid, Mira). The off-diagonal mass is concentrated in three pairs:
W Virginis -> Cepheid (the Pop I / Pop II Cepheid confusion), Beta Cephei
-> Delta Scuti (the short-period hot-star confusion), and SX Phoenicis
-> Delta Scuti (the metal-poor short-period confusion). Each of these
has a specific astrophysical cause that the SVM's global decision
boundary cannot fully resolve.

### Step 5: Support Vectors

In [ ]:
# Listing 19.
n_sv = clf_final.n_support_
sv_classes = clf_final.classes_
total_sv = sum(n_sv)
n_train = len(y_train)

print(f'Total support vectors: {total_sv} / {n_train} training samples '
      f'({100*total_sv/n_train:.1f}%)\n')
print('Support vectors per class:')
for cls, nsv in zip(sv_classes, n_sv):
    n_cls = int(np.sum(y_train == cls))
    print(f'  {CLASS_NAMES[cls]:14s}: {nsv:4d} SV  '
          f'({100*nsv/n_cls:.1f}% of class training samples)')

fig, ax = plt.subplots(figsize=(9, 4))
ax.bar([CLASS_NAMES[c] for c in sv_classes], n_sv, color='steelblue')
ax.set_xlabel('Star type', fontsize=11)
ax.set_ylabel('Number of support vectors', fontsize=11)
ax.set_title('Support Vectors per Class', fontsize=12)
plt.xticks(rotation=30, ha='right')
plt.tight_layout()
plt.show()

A high fraction of support vectors relative to class training size indicates
a class with an ambiguous, wide-margin-violating boundary. W Virginis and
SX Phoenicis typically have the highest support vector fractions: the SVM
needs many training points near the boundary to define the separation,
which itself signals that the separation is poor.

A large fraction of the total training set becoming support vectors (often
30-50% for this dataset) is a sign that the problem is genuinely hard for
an SVM: the margins are narrow everywhere and many points violate them.
This contrasts with easier datasets where 5-15% of points are support
vectors and the margins are wide.

### Where to Go From Here

The honest conclusion from this challenge is that SVMs are not the best
choice for this specific classification problem. The results are informative
and the preprocessing is correct, but the architecture of the algorithm
works against the structure of the data in two ways.

First, small disjuncts. SVMs define global decision boundaries via a
kernel. When a minority class (SX Phe, 4%) is embedded within a majority
class (Delta Scuti, 27%) and differs only in one noisy feature, the SVM
cannot carve out the minority region without either misclassifying most
of the surrounding majority or using an extremely small kernel bandwidth
that overfits noise. Tree-based ensemble methods like random forests
handle this better because each tree is local: it can learn a branch
that applies only in the low-metallicity, short-period region without
affecting decisions elsewhere in the feature space.

Second, class imbalance. With 27% Delta Scuti and 4% SX Phe, the SVM's
loss function is dominated by Delta Scuti misclassifications. The `class_weight`
parameter in `SVC` can partially address this by up-weighting minority
class errors, and oversampling approaches (SMOTE, ADASYN) can augment
the minority training set. These techniques are covered in
Notebook\_7\_Bonus.ipynb.

A random forest or gradient boosted classifier trained on this same
feature set typically achieves 85-91% accuracy on this dataset, with
substantially better minority-class recall, primarily because of the
local decision structure and implicit handling of feature interactions.
The appropriate next experiment after this challenge is to try one of
those alternatives and compare the confusion matrices directly.

---
## 7. References

1. Cortes, C. and Vapnik, V. (1995). Support-vector networks.
   *Machine Learning*, 20(3), 273-297.
   https://doi.org/10.1007/BF00994018
   Original paper introducing the support vector machine.

2. Eyer, L. et al. (2023). Gaia Data Release 3: Summary of the variability
   processing and analysis. *Astronomy and Astrophysics*, 674, A13.
   https://doi.org/10.1051/0004-6361/202244242
   Describes the automated variability classification of 10.5 million
   sources in Gaia DR3, using features similar to those in this dataset.

3. Soszynski, I. et al. (2008). The Optical Gravitational Lensing Experiment.
   Variable stars in the Magellanic Clouds. *Acta Astronomica*, 58, 163.
   Reference for the OGLE variable star catalogues that motivated the
   class distribution and feature set in this challenge.

4. McNamara, D.H. (2011). Pulsation Frequencies and Modes of Giant
   Exoplanets. *The Astrophysical Journal*, 736, 69.
   https://doi.org/10.1088/0004-637X/736/2/69
   Discusses the Delta Scuti / SX Phoenicis classification problem and
   the role of metallicity as a discriminator.

5. Beaulieu, J.P. et al. (2001). Separation of W Virginis and classical
   Cepheids in the Large Magellanic Cloud.
   *Astronomy and Astrophysics*, 380, 168-176.
   https://doi.org/10.1051/0004-6361:20010993
   Documents the Cepheid / W Virginis confusion problem in real survey data.

6. Dubath, P. et al. (2011). Random forest automated supervised
   classification of Hipparcos periodic variable stars.
   *Monthly Notices of the Royal Astronomical Society*, 414(3), 2602-2617.
   https://doi.org/10.1111/j.1365-2966.2011.18575.x
   Benchmark paper comparing SVM and random forest classifiers on
   variable star data from Hipparcos. Random forests outperform SVMs
   on the same minority class problem explored in this challenge.

7. Pedregosa, F. et al. (2011). Scikit-learn: Machine Learning in Python.
   *Journal of Machine Learning Research*, 12, 2825-2830.
   https://jmlr.org/papers/v12/pedregosa11a.html